# ♻️ Campus Waste Segregation Assistant — corrected training notebook

**What this notebook fixes compared with the original one**

| # | Problem in the old notebook | Fix here |
|---|---|---|
| 1 | Validation and test sets were made with `take()` / `skip()` on a reshuffled dataset, so they overlapped | One stratified 70/15/15 split of file paths, made **once**, saved to `splits.csv` |
| 2 | The "fixed" test set came from a *different* split than the training set, so it contained training images | Train, val and test all come from the same split |
| 3 | `shuffle=False` + `validation_split` takes the *last* files of the sorted folder list (only plastic/trash) | Not used any more |
| 4 | Head trained only 10 epochs at lr 1e-4 | Two proper phases: head (lr 1e-3), then fine-tune (lr 3e-5) |
| 5 | Class imbalance ignored (trash = 137 images) | `class_weight` |
| 6 | Weak augmentation, no brightness/contrast | Stronger augmentation |
| 7 | Resize and loading differed between Colab and the app | **One** `load_image()` function used in both places |
| 8 | Preprocessing hidden inside the model but also applied in `app.py` (the bug you saw) | Model takes **raw 0-255 pixels**. App must never call `preprocess_input` |
| 9 | No real-world test | Optional `my_test_photos/` folder evaluated separately |
| 10 | No save-and-reload check | Reload test after saving |

In [ ]:
!nvidia-smi
import tensorflow as tf, keras, sklearn, PIL

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)
print("Pillow    :", PIL.__version__)
print("GPU       :", tf.config.list_physical_devices("GPU"))
# If GPU is empty: Runtime > Change runtime type > T4 GPU, then run again.

## 1. Upload and extract the dataset

In [ ]:
import os, shutil, zipfile
from google.colab import files

if not os.path.exists("dataset-resized.zip"):
    files.upload()          # choose dataset-resized.zip

if os.path.exists("trashnet_dataset"):
    shutil.rmtree("trashnet_dataset")

with zipfile.ZipFile("dataset-resized.zip") as z:
    z.extractall("trashnet_dataset")

shutil.rmtree("trashnet_dataset/__MACOSX", ignore_errors=True)   # Mac junk files
print(os.listdir("trashnet_dataset/dataset-resized"))

## 2. Configuration and imports

**Optional but strongly recommended:** to make the model work on real campus photos, upload your own pictures as
`campus_photos/<class>/xxx.jpg` (for example `campus_photos/plastic/bottle1.jpg`). They are mixed into the training data.
Keep a *separate* set of photos in `my_test_photos/<class>/...` that is **never** trained on. It gives you an honest real-world score.

In [ ]:
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)          # python + numpy + tensorflow

CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]   # alphabetical order
IMG_SIZE    = 224
BATCH_SIZE  = 32
IMG_EXT     = (".jpg", ".jpeg", ".png", ".webp")

DATA_DIR          = "trashnet_dataset/dataset-resized"
EXTRA_DIR         = "campus_photos"        # optional: your own training photos
REALWORLD_TEST_DIR = "my_test_photos"      # optional: your own photos, NEVER trained on

## 3. One shared image-loading function

`app.py` uses exactly the same function, so training and the app see identical inputs.
It returns **raw pixels 0-255**. The model rescales them internally.

In [ ]:
def load_image(source):
    """Return a (224, 224, 3) uint8 array with RAW pixel values 0-255."""
    img = Image.open(source)
    img = ImageOps.exif_transpose(img)                     # fix rotated phone photos
    if img.mode in ("RGBA", "LA", "P"):                    # transparent PNG -> white background
        img = img.convert("RGBA")
        background = Image.new("RGBA", img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(background, img)
    img = img.convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)


def collect_files(root):
    items = []
    for label, name in enumerate(CLASS_NAMES):
        folder = os.path.join(root, name)
        if not os.path.isdir(folder):
            continue
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith(IMG_EXT) and not fname.startswith("._"):
                items.append((os.path.join(folder, fname), label))
    return items

## 4. One clean stratified split (70% train / 15% validation / 15% test)

In [ ]:
items = collect_files(DATA_DIR)
print("TrashNet images     :", len(items))

if os.path.isdir(EXTRA_DIR):
    extra = collect_files(EXTRA_DIR)
    print("Extra campus photos :", len(extra))
    items += extra

paths  = np.array([p for p, _ in items])
labels = np.array([l for _, l in items])

p_train, p_tmp, y_train, y_tmp = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_val, p_test, y_val, y_test = train_test_split(
    p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

# Safety check: the three sets must never share a file
assert not (set(p_train) & set(p_val))
assert not (set(p_train) & set(p_test))
assert not (set(p_val) & set(p_test))

print(f"\n{'class':10} {'train':>6} {'val':>6} {'test':>6}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:10} {(y_train == i).sum():6d} {(y_val == i).sum():6d} {(y_test == i).sum():6d}")
print(f"{'TOTAL':10} {len(y_train):6d} {len(y_val):6d} {len(y_test):6d}")

pd.DataFrame({
    "path": np.concatenate([p_train, p_val, p_test]),
    "label": np.concatenate([y_train, y_val, y_test]),
    "split": ["train"] * len(p_train) + ["val"] * len(p_val) + ["test"] * len(p_test),
}).to_csv("splits.csv", index=False)

## 5. Load images into memory

In [ ]:
def load_many(path_list):
    return np.stack([load_image(p) for p in tqdm(path_list)])

X_train = load_many(p_train)
X_val   = load_many(p_val)
X_test  = load_many(p_test)

print(X_train.shape, X_train.dtype, "pixel range:", X_train.min(), "-", X_train.max())   # expect 0 - 255

## 6. Class weights and tf.data pipelines

In [ ]:
cw = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", {CLASS_NAMES[i]: round(w, 2) for i, w in class_weight.items()})


def make_dataset(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds   = make_dataset(X_val,   y_val,   training=False)
test_ds  = make_dataset(X_test,  y_test,  training=False)

## 7. Model

The model **expects raw pixels 0-255** and rescales them to [-1, 1] inside (`Rescaling` = MobileNetV2's own preprocessing).
Never call `preprocess_input` outside the model.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.25, value_range=(0, 255)),
    layers.RandomContrast(0.25),
], name="augmentation")


def build_model():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")       # raw 0-255
    x = augmentation(inputs)                                                 # only active while training
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_rescale")(x)   # 0-255 -> -1..1

    base = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="waste_classifier"), base

model, base_model = build_model()
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print("Base layers :", len(base_model.layers))

### Quick look at the augmentation (sanity check)

In [ ]:
plt.figure(figsize=(14, 6))
for i in range(8):
    aug = augmentation(X_train[i].astype("float32")[None], training=True)[0].numpy().astype("uint8")
    plt.subplot(2, 4, i + 1)
    plt.imshow(aug)
    plt.title(CLASS_NAMES[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Phase 1 — train the new classifier head (base frozen)

In [ ]:
def make_callbacks(tag):
    return [
        keras.callbacks.ModelCheckpoint(f"best_{tag}.weights.h5", monitor="val_loss",
                                        save_best_only=True, save_weights_only=True),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    ]

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

hist1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=20, class_weight=class_weight,
    callbacks=make_callbacks("phase1"),
)
model.load_weights("best_phase1.weights.h5")          # go back to the best epoch
print("Phase 1 validation:", model.evaluate(val_ds, verbose=0))

## 9. Phase 2 — fine-tune the top of MobileNetV2 (low learning rate)

In [ ]:
FINE_TUNE_FROM = 100                                   # MobileNetV2 has 154 layers

base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False
for layer in base_model.layers:                        # keep BatchNorm statistics stable
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

n_trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
print("Trainable parameters:", f"{n_trainable:,}")

model.compile(                                         # must recompile after changing trainable
    optimizer=keras.optimizers.Adam(3e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

hist2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=25, class_weight=class_weight,
    callbacks=make_callbacks("phase2"),
)
model.load_weights("best_phase2.weights.h5")
print("Phase 2 validation:", model.evaluate(val_ds, verbose=0))

## 10. Learning curves

In [ ]:
acc      = hist1.history["accuracy"]     + hist2.history["accuracy"]
val_acc  = hist1.history["val_accuracy"] + hist2.history["val_accuracy"]
loss     = hist1.history["loss"]         + hist2.history["loss"]
val_loss = hist1.history["val_loss"]     + hist2.history["val_loss"]
split_at = len(hist1.history["accuracy"]) - 0.5

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, tr, va, title in [(ax[0], acc, val_acc, "Accuracy"), (ax[1], loss, val_loss, "Loss")]:
    a.plot(tr, label="train"); a.plot(va, label="validation")
    a.axvline(split_at, color="gray", linestyle="--", label="fine-tuning starts")
    a.set_title(title); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 11. Final evaluation on the untouched test set

In [ ]:
probs_test = model.predict(X_test.astype("float32"), batch_size=BATCH_SIZE, verbose=0)
pred_test  = probs_test.argmax(axis=1)

print(f"TEST ACCURACY: {(pred_test == y_test).mean() * 100:.2f}%   ({len(y_test)} images)\n")
print(classification_report(y_test, pred_test, target_names=CLASS_NAMES, digits=3))

cm = confusion_matrix(y_test, pred_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (test set)")
plt.tight_layout(); plt.show()

wrong = np.where(pred_test != y_test)[0][:12]
if len(wrong):
    plt.figure(figsize=(14, 7))
    for k, i in enumerate(wrong):
        plt.subplot(3, 4, k + 1)
        plt.imshow(X_test[i])
        plt.title(f"true: {CLASS_NAMES[y_test[i]]}\npred: {CLASS_NAMES[pred_test[i]]} ({probs_test[i].max()*100:.0f}%)", fontsize=9)
        plt.axis("off")
    plt.suptitle("Misclassified test images"); plt.tight_layout(); plt.show()

## 12. Proof of the app bug (double preprocessing)

The old `app.py` ran `preprocess_input` *before* the model, which rescales again internally.
This cell shows what that does to your accuracy.

In [ ]:
from keras.applications.mobilenet_v2 import preprocess_input

correct_in = X_test.astype("float32")                       # raw 0-255  (what the model expects)
double_in  = preprocess_input(X_test.astype("float32"))     # scaled twice (what old app.py did)

acc_ok  = (model.predict(correct_in, verbose=0).argmax(1) == y_test).mean()
acc_bad = (model.predict(double_in,  verbose=0).argmax(1) == y_test).mean()

print(f"Raw pixels (correct)        : {acc_ok  * 100:.2f}%")
print(f"preprocess_input applied 2x : {acc_bad * 100:.2f}%   <- the bug in the old app.py")

## 13. Optional: honest real-world test

Put photos you never trained on in `my_test_photos/<class>/`, for example `my_test_photos/plastic/bottle_on_table.jpg`.

In [ ]:
if os.path.isdir(REALWORLD_TEST_DIR):
    rw = collect_files(REALWORLD_TEST_DIR)
    if rw:
        hits = 0
        for path, true_label in rw:
            p = model.predict(load_image(path).astype("float32")[None], verbose=0)[0]
            pred = int(p.argmax()); hits += (pred == true_label)
            flag = "OK " if pred == true_label else "BAD"
            print(f"{flag} {os.path.basename(path):30} true={CLASS_NAMES[true_label]:9} pred={CLASS_NAMES[pred]:9} {p[pred]*100:5.1f}%")
        print(f"\nReal-world accuracy: {hits / len(rw) * 100:.1f}%  ({len(rw)} photos)")
    else:
        print("my_test_photos/ exists but has no images in the class sub-folders.")
else:
    print("No my_test_photos/ folder. Upload some real photos there to measure real-world accuracy.")

## 14. Save, reload check, and download

In [ ]:
model.save("waste_segregation_mobilenetv2.keras")
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)

# Reload exactly like the app will, and confirm identical predictions
reloaded = keras.models.load_model("waste_segregation_mobilenetv2.keras", compile=False)
a = model.predict(X_test[:32].astype("float32"), verbose=0)
b = reloaded.predict(X_test[:32].astype("float32"), verbose=0)
print("Max difference after reload:", float(np.abs(a - b).max()))     # should be ~0

# Simulate the app on one real file
sample = load_image(p_test[0])
out = reloaded.predict(sample.astype("float32")[None], verbose=0)[0]
print("App-style test ->", CLASS_NAMES[int(out.argmax())], f"{out.max()*100:.1f}%", "| true:", CLASS_NAMES[y_test[0]])

print("\nVersions to use in your local environment (requirements.txt):")
print("  tensorflow ==", tf.__version__)
print("  keras      ==", keras.__version__)

from google.colab import files
files.download("waste_segregation_mobilenetv2.keras")
files.download("class_names.json")
files.download("splits.csv")